# Behind the pipeline

In [1]:
!pip install -q datasets evaluate transformers[sentencepiece]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.9 MB/s eta 0:00:00


In [2]:
from transformers import pipeline

# "sentiment-analysis" 파이프라인을 로드
classifier = pipeline("sentiment-analysis")

# 주어진 문장에 대한 감정 분석 수행
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

# 결과 출력
print(result)

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


### Tokenizer

In [3]:
from transformers import pipeline

# "sentiment-analysis" 파이프라인을 로드
classifier = pipeline("sentiment-analysis")

# 주어진 문장에 대한 감정 분석 수행
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

# 결과 출력
print(result)

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'label': 'POSITIVE', 'score': 0.9598048329353333}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [4]:
# Hugging Face Transformers 라이브러리에서 tokenizer를 가져옴
from transformers import AutoTokenizer

# 원시 입력 데이터
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

# tokenizer 초기화
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 입력 데이터를 처리하고 패딩, 자르기 및 PyTorch 텐서로 변환
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")

# 처리된 입력 데이터 출력
print(inputs)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [5]:
inputs['input_ids'].shape

torch.Size([2, 16])

In [6]:
for input_id in inputs["input_ids"]:
    print(tokenizer.decode(input_id))

[CLS] i've been waiting for a huggingface course my whole life. [SEP]
[CLS] i hate this so much! [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


### Model

tokenizer와 동일한 checkpoint의 모델 다운로드

In [7]:
# AutoModel - hidden state 조회
from transformers import AutoModel

# 미리 훈련된 모델을 로드할 체크포인트 경로 또는 모델 이름 (예: "bert-base-uncased")
checkpoint = "bert-base-uncased"

# 미리 훈련된 모델을 로드
model = AutoModel.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [8]:
# 모델에 입력 데이터(inputs)를 전달하여 출력(outputs)을 얻음
outputs = model(**inputs)

# 출력(outputs)의 마지막 은닉 상태(Hidden State)의 형태(shape)를 출력
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


### Model Head

In [9]:
from transformers import AutoModelForSequenceClassification

# 미리 훈련된 모델을 로드할 체크포인트 경로 또는 모델 이름 (예: "bert-base-uncased")
checkpoint = "bert-base-uncased"

# 미리 훈련된 시퀀스 분류 모델을 로드
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# 모델에 입력 데이터(inputs)를 전달하여 출력(outputs)을 얻음
outputs = model(**inputs)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
print(outputs.logits.shape)

torch.Size([2, 2])


In [11]:
print(outputs.logits)

tensor([[-0.3129, -0.5227],
        [-0.1432, -0.5493]], grad_fn=<AddmmBackward0>)


### Logit 에 Softmax 적용 - post processing

In [12]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[0.5523, 0.4477],
        [0.6002, 0.3998]], grad_fn=<SoftmaxBackward0>)


In [13]:
model.config.id2label

{0: 'LABEL_0', 1: 'LABEL_1'}